<a href="https://colab.research.google.com/github/heisenberg304/aprendiendo-ml-cloud/blob/deep-learning/MNIST_sin_TensorFlow_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Para MNIST usamos una distribución de probabilidades, por lo que he reemplazado MSE, por cross entropy en este caso
#Cross-Entropy está diseñada precisamente para comparar distribuciones de probabilidad
#
import numpy as np
from data_MNIST import x, y
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

x = x[:1000]
y = y[:1000]

x_test = x[800:] #(200, 784)
y_test = y[800:] #(200,)

y_one_hot = np.zeros((y.size, 10))
y_one_hot[np.arange(y.size), y] = 1
y = y_one_hot

x_train = x[:800] #(800, 784)
y_train = y[:800] #(800, 10)

# Pesos de las neuronas de las capas Hidden y output
input_size=784
hidden1=100
hidden2=100
hidden3=50
output_size=10
def activacion_de_pesos(input_size, hidden):
    w = np.random.randn(input_size, hidden).astype(np.float32) * np.sqrt(2.0 / input_size)
    b = np.zeros((1, hidden), dtype=np.float32)
    return w,b
w1, b1 = activacion_de_pesos(input_size, hidden1)
w2, b2 = activacion_de_pesos(hidden1, hidden2)
w3, b3 = activacion_de_pesos(hidden2, hidden3)
wo, bo = activacion_de_pesos(hidden3, output_size)

learning_rate = 0.4


#funciones de activacion y sus gradientes
#Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def grad_sigmoid(z):
    return sigmoid(z) * (1 - sigmoid(z))
#ReLU
def relu(z):
    return np.maximum(0, z)
def grad_relu(z):
    return (z > 0).astype(float)
#Softmax
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    p = exp_z / np.sum(exp_z, axis=1, keepdims=True)
    return p
#La derivada de Softmax sola es su Jacobiano 10*10, pero para simplificar usaremos:
def grad_softmax(p, y):
    return p - y

#para la prediccion de testeo
def predict(w,x,b):
    pred = np.dot(w, x) + b
    return pred

#Cross-entropy
def cross_entropy(p, y):
    epsilon = 1e-15
    p = np.clip(p, epsilon, 1 - epsilon)
    return -np.mean(np.sum(y * np.log(p), axis=1))


#NEURONAS
#implementacion para las neuronas hidden
def hidden_neuron_matrices(w, x, b, funcion_de_activacion=relu):
  #XW+B
    z = np.dot(x, w) + b
    h = funcion_de_activacion(z)
    return z, h
#implementacion para las neuronas output
def output_neuron(w, x, b):
  #XW+B
    z = np.dot(x, w) + b
    h = softmax(z)
    return z, h


#GRADIENTES DE W & B
def gradiente_hidden_w_b(gradz, w, z, h, x, grad_funcion_de_activacion=grad_relu):
    grad_z = gradz @ w.T * grad_funcion_de_activacion(z)
    grad_b = np.mean(grad_z, axis=0)
    grad_w = h.T @ grad_z / x.shape[0]
    return grad_z, grad_b, grad_w

#actualizacion
def actualizar(lr, w_b, grad):
    new = w_b - (lr*grad)
    return new

def main(x, wo, bo, w1, b1, w2, b2, w3, b3, y, funcion_de_activacion=relu, grad_funcion_de_activacion=grad_relu):
    z1, h1 = hidden_neuron_matrices(w1, x, b1, funcion_de_activacion)
    z2, h2 = hidden_neuron_matrices(w2, h1, b2, funcion_de_activacion)
    z3, h3 = hidden_neuron_matrices(w3, h2, b3, funcion_de_activacion)
    zo, output = output_neuron(wo, h3, bo)

    funcion_loss = cross_entropy(output, y)

    grad_zo = grad_softmax(output, y)
    grad_bo = np.mean(grad_zo, axis=0)
    grad_wo = h3.T @ grad_zo / x.shape[0]

    grad_z3, grad_b3, grad_w3 = gradiente_hidden_w_b(grad_zo, wo, z3, h2, x, grad_funcion_de_activacion)
    grad_z2, grad_b2, grad_w2 = gradiente_hidden_w_b(grad_z3, w3, z2, h1, x, grad_funcion_de_activacion)
    grad_z1, grad_b1, grad_w1 = gradiente_hidden_w_b(grad_z2, w2, z1, x, x, grad_funcion_de_activacion)

    wo_new = actualizar(learning_rate, wo, grad_wo)
    bo_new = actualizar(learning_rate, bo, grad_bo)
    w3_new = actualizar(learning_rate, w3, grad_w3)
    b3_new = actualizar(learning_rate, b3, grad_b3)
    w2_new = actualizar(learning_rate, w2, grad_w2)
    b2_new = actualizar(learning_rate, b2, grad_b2)
    w1_new = actualizar(learning_rate, w1, grad_w1)
    b1_new = actualizar(learning_rate, b1, grad_b1)
    return grad_wo, grad_bo, grad_w1, grad_b1, grad_w2, grad_b2, grad_w3, grad_b3, output, funcion_loss, wo_new, bo_new, w3_new, b3_new, w2_new, b2_new, w1_new, b1_new

grad_wo, grad_bo, grad_w1, grad_b1, grad_w2, grad_b2, grad_w3, grad_b3, output, funcion_loss, wo_new, bo_new, w3_new, b3_new, w2_new, b2_new, w1_new, b1_new = main(x_train, wo, bo, w1, b1, w2, b2, w3, b3, y_train, relu, grad_relu)

def metrics(y_test, pred, average_type='binary'):
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, average=average_type, zero_division=0)
    recall = recall_score(y_test, pred, average=average_type, zero_division=0)
    f1 = f1_score(y_test, pred, average=average_type, zero_division=0)
    return accuracy, precision, recall, f1

def test_metrics(w1, b1, w2, b2, w3, b3, wo, bo, y_test, x_test):
    z1, h1 = hidden_neuron_matrices(w1, x_test, b1)
    z2, h2 = hidden_neuron_matrices(w2, h1, b2)
    z3, h3 = hidden_neuron_matrices(w3, h2, b3)
    zo, output = output_neuron(wo, h3, bo)
    pred = np.argmax(output, axis=1)
    accuracy, precision, recall, f1 = metrics(y_test, pred, average_type='macro')
    return accuracy, precision, recall, f1

for epoch in range(1001):
    grad_wo, grad_bo, grad_w1, grad_b1, grad_w2, grad_b2, grad_w3, grad_b3, output, funcion_loss, wo, bo, w3, b3, w2, b2, w1, b1 = main(x_train, wo, bo, w1, b1, w2, b2, w3, b3, y_train, relu, grad_relu)
    if epoch % 100 == 0:
        print(f"Epoch: {epoch}")
        print(f"Loss: {funcion_loss}")
        print(f"Predicción: {np.argmax(output, axis=1)}")

    if epoch % 1000 == 0:
        np.savez(
            "pesos_modelo.npz",
            w1=w1,
            b1=b1,
            w2=w2,
            b2=b2,
            w3=w3,
            b3=b3,
            wo=wo,
            bo=bo
        )
        accuracy, precision, recall, f1 = test_metrics(w1, b1, w2, b2, w3, b3, wo, bo, y_test, x_test)
        print(
            f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1 Score: {f1}"
        )



Epoch: 0
Loss: 2.3726485794362873
Predicción: [4 9 1 7 7 7 7 7 7 7 7 4 7 4 4 9 7 7 7 9 9 4 7 7 9 7 7 7 7 7 7 7 7 4 7 4 7
 9 7 7 4 7 4 7 7 4 4 9 1 7 9 9 4 7 7 4 4 7 7 7 7 7 7 9 1 1 7 7 9 7 4 7 4 4
 7 7 7 7 7 7 9 9 9 4 4 7 4 4 4 4 4 7 7 7 4 9 4 4 7 7 4 7 7 2 4 7 7 4 4 1 4
 7 7 7 7 7 7 4 9 1 9 7 4 9 7 7 7 4 7 7 7 4 7 7 7 1 7 7 7 7 7 7 7 4 7 6 7 7
 7 7 1 7 7 4 7 4 4 1 4 4 7 4 1 7 7 4 7 0 7 7 7 4 7 7 7 6 7 7 7 7 7 4 7 7 4
 9 7 7 4 7 7 9 9 7 7 7 7 7 7 4 7 7 9 7 7 7 4 7 7 7 4 7 7 7 7 1 7 7 4 4 7 4
 4 9 7 7 7 7 7 0 4 7 7 4 7 7 9 7 4 7 3 7 7 2 1 7 4 7 7 4 7 7 7 9 7 9 7 7 4
 7 7 4 4 9 4 7 1 4 4 7 7 4 7 7 7 7 7 9 7 1 7 4 7 9 7 7 7 7 7 7 7 7 7 4 4 4
 4 7 7 4 7 7 7 9 1 9 7 4 9 7 7 4 7 4 7 7 7 4 1 7 7 7 4 4 7 7 9 9 7 7 7 7 2
 4 7 4 7 9 7 7 7 4 4 4 4 7 7 9 7 9 7 7 0 4 7 7 7 7 4 9 7 7 7 4 4 7 7 4 7 7
 4 4 7 7 7 4 2 9 9 4 9 4 7 4 7 6 1 7 7 7 7 7 7 4 7 7 7 7 4 9 1 7 7 4 4 4 7
 4 7 7 7 7 4 7 7 4 7 7 7 7 4 4 4 7 4 7 7 9 7 4 7 4 7 1 7 7 7 4 7 4 4 7 7 7
 7 7 4 7 7 4 7 9 7 4 4 7 2 4 9 1 7 4 9 9 7 4 7 4 9 4 7

In [2]:
%%writefile data_MNIST.py
import numpy as np
from sklearn.datasets import fetch_openml
X, y = fetch_openml(
    "mnist_784",
    version=1,
    return_X_y=True,
    as_frame=False
)
x = X.astype(np.float32) / 255.0
y = y.astype(np.int64)


Writing data_MNIST.py


In [ ]:
#
#importar librerías
import numpy as np

# Entrada
x = np.array([[   #(1, 25)
    0, 0, 1, 0, 0,
    0, 1, 1, 1, 0,
    1, 1, 1, 1, 1,
    0, 1, 1, 1, 0,
    0, 0, 1, 0, 0
]])
y = np.array([[   #(1, 10)
    0, 0, 0, 1, 0,
    0, 0, 0, 0, 0
]])

# Pesos de las 3 neuronas Hidden
w1 = np.array([    #(25, 5)
    [ 0.2, -0.1,  0.4,  0.3, -0.2],
    [-0.3,  0.5,  0.1, -0.4,  0.2],
    [ 0.6,  0.2, -0.5,  0.1,  0.3],
    [-0.2,  0.4,  0.3,  0.6, -0.1],
    [ 0.1, -0.5,  0.2, -0.3,  0.7],
    [ 0.4,  0.3, -0.2,  0.5,  0.1],
    [-0.1,  0.6,  0.4, -0.2,  0.3],
    [ 0.5, -0.3,  0.1,  0.2, -0.4],
    [ 0.3,  0.2,  0.5, -0.1,  0.6],
    [-0.4,  0.1, -0.3,  0.4,  0.2],
    [ 0.2,  0.7,  0.1, -0.5,  0.3],
    [-0.5,  0.2,  0.4,  0.3, -0.1],
    [ 0.3, -0.4,  0.6,  0.2,  0.1],
    [ 0.1,  0.5, -0.2,  0.4, -0.3],
    [-0.2,  0.3,  0.5, -0.1,  0.6],
    [ 0.4, -0.2,  0.1,  0.5,  0.3],
    [-0.3,  0.6,  0.2, -0.4,  0.1],
    [ 0.5,  0.1, -0.3,  0.2,  0.4],
    [ 0.2, -0.5,  0.3,  0.6, -0.1],
    [-0.1,  0.4,  0.2, -0.3,  0.5],
    [ 0.6,  0.2, -0.4,  0.1,  0.3],
    [-0.2,  0.5,  0.3, -0.1,  0.4],
    [ 0.3, -0.3,  0.1,  0.5,  0.2],
    [ 0.1,  0.4, -0.2,  0.3, -0.5],
    [ 0.5, -0.1,  0.2,  0.4,  0.3]
])
w2 = np.array([     #(5, 5)
    [ 0.2, -0.4,  0.5,  0.1,  0.3],
    [-0.3,  0.6,  0.2, -0.5,  0.4],
    [ 0.7,  0.1, -0.2,  0.3, -0.1],
    [ 0.4, -0.2,  0.6,  0.5,  0.1],
    [-0.1,  0.3,  0.4, -0.3,  0.5]
])
w3 = np.array([     #(5, 5)
    [ 0.3, -0.2,  0.5,  0.1, -0.4],
    [ 0.6,  0.1, -0.3,  0.4,  0.2],
    [-0.2,  0.5,  0.3, -0.1,  0.6],
    [ 0.4, -0.5,  0.2,  0.3,  0.1],
    [ 0.1,  0.3, -0.4,  0.6,  0.5]
])
# Bias de las 3 neuronas Hidden   (5,)
b1 = np.array([0.1, -0.2, 0.3, 0.05, -0.1])
b2 = np.array([-0.1, 0.2, 0.05, -0.3, 0.15])
b3 = np.array([0.2, -0.15, 0.1, 0.3, -0.05])

# Pesos de Hidden → Output
wo = np.array([    #(5, 10)
    [ 0.2, -0.1,  0.4,  0.3, -0.2,  0.5,  0.1, -0.3,  0.6,  0.2],
    [-0.4,  0.5,  0.2, -0.1,  0.3, -0.2,  0.6,  0.4, -0.5,  0.1],
    [ 0.3,  0.2, -0.3,  0.6,  0.1,  0.4, -0.2,  0.5,  0.3, -0.1],
    [ 0.5, -0.3,  0.1,  0.4, -0.2,  0.3,  0.6, -0.1,  0.2,  0.5],
    [-0.1,  0.4,  0.5, -0.2,  0.3,  0.1, -0.4,  0.6,  0.2, -0.3]
])
# Bias del Output
bo = np.array([    #(10,)
    0.1, -0.2, 0.05, 0.3, -0.1,
    0.2, -0.15, 0.25, 0.05, -0.3
])

learning_rate = 0.1

#funciones de activacion y sus gradientes
def sigmoid(z):
  #Sigmoid
  return 1 / (1 + np.exp(-z))
def grad_sigmoid(z):
  return sigmoid(z) * (1 - sigmoid(z))

def softmax(z):
  #softmax
  exp_z = np.exp(z - np.max(z))
  p = exp_z / np.sum(exp_z)
  return p
#La derivada de Softmax sola es su Jacobiano 10*10, pero como esta con cross-entropy usaremos para simplificar:
def grad_softmax(p, y):
  return p - y

#predicciones y error
def predict(w,x,b):
  pred = np.dot(w, x) + b
  return pred
def error(pred, y):
  error = pred - y
  return error

#funcion L
def f_loss(error):
  #MSE
  L = np.mean(error ** 2) / 2
  return L
def cross_entropy(p, y):
  #Cross-
  L = -np.sum(y * np.log(p))
  return L


#NEURONAS
#implementacion para las neuronas hidden
def hidden_neuron_matrices(w, x, b):
  #XW+B
  z = np.dot(x, w) + b
  h = sigmoid(z)
  return z, h
#implementacion para las neuronas output
def output_neuron(w, x, b):
  #XW+B
  z = np.dot(x, w) + b
  h = softmax(z)
  return z, h

#grad_b3= grad_bo*wo*grad_sigmoid(z3) =
#        (10,) * (5,10) * (1,5)  ->  (1,5) -> (5,)
#grad_b2= grad_b3*w3*grad_sigmoid(z2) =
#          (5,) * (5,5) * (1,5)  ->  (1,5) -> (5,)
#grad_b1= grad_b2*w2*grad_sigmoid(z1) =
#         (5,) * (25,5) * (1,5)  ->  (1,5) -> (5,)

#GRADIENTES DE B
#recuerda la forma del gradiente del peso debe ser igual a la forma del peso
#el backpropagation sera: output -> h3 -> h2 -> h1
def gradiente_hidden_b(h, z):
  grad_hidden = h * grad_sigmoid(z)
  return grad_hidden

#actualizacion
def actualizar(lr, w_b, grad):
  new = w_b - (lr*grad)
  return new

                            #w1 = (25, 5) , b1 = (5,)
z1, h1 = hidden_neuron_matrices(w1, x, b1) #z1(1, 5) h1(1, 5)
                            #w2 = (5, 5) , b2 = (5,)
z2, h2 = hidden_neuron_matrices(w2, h1, b2) #z2(1, 5) h2(1, 5)
                            #w3 = (5, 5) , b3 = (5,)
z3, h3 = hidden_neuron_matrices(w3, h2, b3) #z3(1, 5) h3(1, 5)
  #output                   #wo = (5, 10) , bo = (10,)
zo, output = output_neuron(wo, h3, bo) #zo = (1, 10) , output = (1, 10)
#IMPORTANTE: El gradiente respecto a los pesos debe tener la misma dimensión que los peso: shape(grad_w) = shape(w)

funcion_loss = cross_entropy(output, y)

#grad_bo = grad_zo
grad_zo = grad_softmax(output, y) #(1,10)
#shape(grad_bo) = shape(bo)
grad_bo = np.sum(grad_zo, axis=0) #(10,)
#shape(grad_wo) = shape(wo)
grad_wo = h3.T @ grad_zo #(5,10)

#grad_b3 = grad_z3
grad_z3 = grad_zo @ wo.T * grad_sigmoid(z3) #(1,5)
#shape(grad_b3) = shape(b3)
grad_b3 = np.sum(grad_z3, axis=0) #(5,)
#shape(grad_w3) = shape(w3)
grad_w3 = h2.T @ grad_z3 #(5, 5)

#grad_b2 = grad_z2
grad_z2 = grad_z3 @ w3.T * grad_sigmoid(z2) #(1,5)
#shape(grad_b2) = shape(b2)
grad_b2 = np.sum(grad_z2, axis=0) #(5,)
#shape(grad_w2) = shape(w2)
grad_w2 = h1.T @ grad_z2 #(5, 5)

#grad_b1 = grad_z1
grad_z1 = grad_z2 @ w2.T * grad_sigmoid(z1) #(1, 5)
#shape(grad_b1) = shape(b1)
grad_b1 = np.sum(grad_z1, axis=0) #(5,)
#shape(grad_w1) = shape(w1)
grad_w1 = x.T @ grad_z1 #(25, 5)


grads = (grad_wo.shape, grad_bo.shape, grad_w1.shape, grad_b1.shape, grad_w2.shape, grad_b2.shape, grad_w3.shape, grad_b3.shape)
print("grads", grads)

print("r", y.shape, x.shape)
print(z1, z2.shape, z3)
#     (1, 5)     (1, 5)     (1, 5)
print(h1.shape, h2.shape, h3.shape)
print(zo.shape, output.shape)
#print(w1.shape,w2.shape,w3.shape,wo.shape)
#print(b1.shape,b2.shape,b3.shape,bo.shape)
#print(x.shape,y.shape)





#

grads ((5, 10), (10,), (25, 5), (5,), (5, 5), (5,), (5, 5), (5,))
r (1, 10) (1, 25)
[[2.   1.7  2.5  1.25 1.9 ]] (1, 5) [[0.87759209 0.17981529 0.26121151 1.13322256 0.70676675]]
(1, 5) (1, 5) (1, 5)
(1, 10) (1, 10)
